# Situated Learning – Evidence demo: on‑premise Machine Translation for automotive documents
**Group 81 · AIML ZG519 Assignment 2, Part 2 (Topic 7 – Machine Translation)**

**Purpose.** In our automotive organisation, German and Japanese customer documents (customer‑specific requirements,
vehicle specifications) are translated to English. Confidentiality agreements forbid uploading them to public / API‑based
translators or LLMs. This notebook demonstrates the **proposed improvement**: open‑source neural MT models that are
copied once to an internal folder and then run **fully offline on a CPU**, so no text leaves the machine.

> **All sentences below are synthetic**, written for this demo in the style of automotive requirement documents.
> They contain no customer or company information. Reference translations were prepared by the group (AI‑assisted)
> for evaluation only.

**Run:** Colab, *CPU runtime is enough* (`Runtime → Run all`, ≈ 5–10 min, mostly model download).

In [ ]:
!pip install -q transformers sentencepiece sacremoses sacrebleu

In [ ]:
import os, re, time, platform
import pandas as pd, torch, sacrebleu, transformers
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
torch.manual_seed(42)
DEVICE = "cpu"            # deliberately CPU: an on-premise laptop/server without GPU
print("transformers", transformers.__version__, "| sacrebleu", sacrebleu.__version__, "| torch", torch.__version__)
print("CPU threads:", torch.get_num_threads(), "|", platform.processor() or platform.machine())
pd.set_option("display.max_colwidth", 200)

## 1. Synthetic test set (German → English, Japanese → English)

In [ ]:
DE = [
 ("Das Steuergerät muss bei einer Umgebungstemperatur von -40 °C bis +85 °C fehlerfrei funktionieren.",
  "The control unit shall operate without errors at an ambient temperature of -40 °C to +85 °C."),
 ("Die Bremsanlage darf bei einem Ausfall des Bordnetzes nicht vollständig abgeschaltet werden.",
  "The braking system must not be completely switched off in the event of a failure of the vehicle electrical system."),
 ("Der Lieferant muss alle Änderungen am Produkt dem Kunden spätestens sechs Wochen vor Serienstart mitteilen.",
  "The supplier shall notify the customer of all changes to the product no later than six weeks before the start of series production."),
 ("Das maximale Drehmoment des Elektromotors beträgt 320 Nm bei 4.500 Umdrehungen pro Minute.",
  "The maximum torque of the electric motor is 320 Nm at 4,500 revolutions per minute."),
 ("Alle sicherheitsrelevanten Bauteile sind gemäß ISO 26262 zu entwickeln und zu dokumentieren.",
  "All safety-relevant components shall be developed and documented in accordance with ISO 26262."),
 ("Die Dauerlaufprüfung ist mit mindestens 1.000 Betriebsstunden durchzuführen.",
  "The endurance test shall be carried out with at least 1,000 operating hours."),
 ("Das Lastenheft gilt in Verbindung mit den kundenspezifischen Anforderungen in der jeweils gültigen Fassung.",
  "The requirements specification applies in conjunction with the customer-specific requirements in the currently valid version."),
 ("Bei Überschreitung der zulässigen Spannung muss das System innerhalb von 10 ms in einen sicheren Zustand wechseln.",
  "If the permissible voltage is exceeded, the system shall switch to a safe state within 10 ms."),
]
JA = [
 ("制御ユニットは、-40℃から+85℃の周囲温度で正常に動作しなければならない。",
  "The control unit shall operate normally at an ambient temperature of -40 °C to +85 °C."),
 ("サプライヤーは、量産開始の6週間前までに、製品のすべての変更を顧客に通知すること。",
  "The supplier shall notify the customer of all changes to the product at least six weeks before the start of mass production."),
 ("本仕様書に記載のない事項については、顧客と協議の上決定する。",
  "Matters not described in this specification shall be decided after consultation with the customer."),
 ("耐久試験は、1,000時間以上実施すること。",
  "The durability test shall be conducted for 1,000 hours or more."),
 ("電圧が許容値を超えた場合、システムは10ミリ秒以内に安全な状態へ移行しなければならない。",
  "If the voltage exceeds the permissible value, the system shall transition to a safe state within 10 milliseconds."),
 ("ブレーキシステムの部品は、すべて機能安全規格ISO 26262に従って開発すること。",
  "All components of the brake system shall be developed in accordance with the functional safety standard ISO 26262."),
]
# Domain glossary: source term -> acceptable English renderings (what a terminology database would enforce)
GLOSSARY = {
 "de": {"Steuergerät": ["control unit", "ecu", "control device"], "Bremsanlage": ["braking system", "brake system"],
        "Bordnetz": ["electrical system", "on-board power", "on-board network", "power supply", "vehicle electrical"],
        "Serienstart": ["start of series production", "start of production", "sop", "series production"],
        "Drehmoment": ["torque"], "sicherheitsrelevanten": ["safety-relevant", "safety relevant", "safety-related"],
        "Dauerlaufprüfung": ["endurance test", "durability test", "endurance run"],
        "Lastenheft": ["requirements specification", "specification", "requirement specification"],
        "kundenspezifischen Anforderungen": ["customer-specific requirements", "customer specific requirements"],
        "sicheren Zustand": ["safe state"]},
 "ja": {"制御ユニット": ["control unit", "ecu"], "量産開始": ["mass production", "start of production", "series production"],
        "仕様書": ["specification"], "耐久試験": ["durability test", "endurance test"], "安全な状態": ["safe state"],
        "ブレーキシステム": ["brake system", "braking system"], "機能安全": ["functional safety"],
        "サプライヤー": ["supplier"], "顧客": ["customer"]},
}
NUMBER_RE = re.compile(r"\d+(?:[.,]\d+)?")
print(len(DE), "German and", len(JA), "Japanese sentences;",
      sum(len(v) for v in GLOSSARY.values()), "glossary terms")

## 2. Copy the models once into an internal folder, then translate **offline**
`save_pretrained` writes the weights to `./onprem_models/…`; every translation below loads them with
`local_files_only=True`, i.e. no network access and nothing sent to any external service.

In [ ]:
MODELS = {
  "OPUS-MT de-en (Marian, 74M)": ("Helsinki-NLP/opus-mt-de-en", "de", {}),
  "OPUS-MT ja-en (Marian, 74M)": ("Helsinki-NLP/opus-mt-ja-en", "ja", {}),
  "NLLB-200 distilled 600M (de)": ("facebook/nllb-200-distilled-600M", "de", {"src_lang": "deu_Latn"}),
  "NLLB-200 distilled 600M (ja)": ("facebook/nllb-200-distilled-600M", "ja", {"src_lang": "jpn_Jpan"}),
}
LOCAL = {}
for name, (hub_id, _, _) in MODELS.items():
    path = os.path.join("onprem_models", hub_id.replace("/", "__"))
    if not os.path.isdir(path):                       # one-time download (would be done by IT, then air-gapped)
        AutoTokenizer.from_pretrained(hub_id).save_pretrained(path)
        AutoModelForSeq2SeqLM.from_pretrained(hub_id).save_pretrained(path)
    LOCAL[name] = path
for p in sorted(set(LOCAL.values())):
    size = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(p) for f in fs) / 1e6
    print(f"{p:<55} {size:8.0f} MB on disk")

In [ ]:
def translate(name, sentences, bs=4):
    path = LOCAL[name]; _, lang, kw = MODELS[name]
    tok = AutoTokenizer.from_pretrained(path, local_files_only=True, **kw)
    model = AutoModelForSeq2SeqLM.from_pretrained(path, local_files_only=True).to(DEVICE).eval()
    gen_kw = {"max_new_tokens": 128, "num_beams": 4}
    if "src_lang" in kw:                                # NLLB: force English as the target language
        gen_kw["forced_bos_token_id"] = tok.convert_tokens_to_ids("eng_Latn")
    out, t0 = [], time.perf_counter()
    for i in range(0, len(sentences), bs):
        enc = tok(sentences[i:i + bs], return_tensors="pt", padding=True).to(DEVICE)
        with torch.no_grad():
            ids = model.generate(**enc, **gen_kw)
        out += tok.batch_decode(ids, skip_special_tokens=True)
    sec_per_sent = (time.perf_counter() - t0) / len(sentences)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    del model
    return out, sec_per_sent, n_params

def glossary_hits(src, hyp, lang):
    hits, total, missed = 0, 0, []
    for term, options in GLOSSARY[lang].items():
        if term in src:
            total += 1
            if any(o in hyp.lower() for o in options):
                hits += 1
            else:
                missed.append(term)
    return hits, total, missed

WORDS = {"one": "1", "two": "2", "three": "3", "four": "4", "five": "5", "six": "6", "seven": "7", "eight": "8",
         "nine": "9", "ten": "10", "thousand": "1000"}
def numbers_ok(src, hyp):
    # Every number in the source must appear in the output (digits, or a spelled-out English number word).
    norm = lambda s: {n.replace(".", "").replace(",", "") for n in NUMBER_RE.findall(s)}
    hyp_nums = norm(hyp) | {v for w, v in WORDS.items() if re.search(rf"\b{w}\b", hyp, re.I)}
    return norm(src) <= hyp_nums

results, rows = [], []
for name, (_, lang, _) in MODELS.items():
    data = DE if lang == "de" else JA
    src = [s for s, _ in data]; ref = [r for _, r in data]
    hyp, sps, n_params = translate(name, src)
    g_hit = g_tot = 0; num_ok = 0
    for s, h, r in zip(src, hyp, ref):
        h_, t_, missed = glossary_hits(s, h, lang); g_hit += h_; g_tot += t_
        ok = numbers_ok(s, h); num_ok += ok
        rows.append({"model": name, "source": s, "MT output": h, "reference": r,
                     "missed glossary terms": ", ".join(missed), "numbers preserved": ok})
    results.append({"model": name, "pair": f"{lang}→en", "params (M)": round(n_params),
                    "BLEU": sacrebleu.corpus_bleu(hyp, [ref]).score,
                    "chrF": sacrebleu.corpus_chrf(hyp, [ref]).score,
                    "glossary terms correct": f"{g_hit}/{g_tot}",
                    "numbers/units preserved": f"{num_ok}/{len(src)}",
                    "CPU s/sentence": sps})
summary = pd.DataFrame(results).round(2)
summary

## 3. Side‑by‑side outputs (for the evidence section)

In [ ]:
detail = pd.DataFrame(rows)
detail.to_csv("mt_demo_outputs.csv", index=False)
for name in MODELS:
    print("=" * 20, name)
    display(detail[detail.model == name][["source", "MT output", "reference", "missed glossary terms", "numbers preserved"]])

In [ ]:
# Requirement-language check: obligations ("muss / ist ... zu" / "しなければならない / こと") must stay obligations,
# and the prohibition in DE sentence 2 must stay a prohibition – a mistranslated modal changes a contractual requirement.
OBLIG = re.compile(r"\b(shall|must|has to|have to|is to|are to|required|needs? to|should)\b", re.I)
checks = []
for name, (_, lang, _) in MODELS.items():
    sub = detail[detail.model == name]
    oblig = [bool(OBLIG.search(h)) for h in sub["MT output"]]
    checks.append({"model": name, "sentences": len(sub), "obligation/modal kept": f"{sum(oblig)}/{len(sub)}"})
    if lang == "de":
        neg = sub["MT output"].iloc[1]
        checks[-1]["DE #2 prohibition kept ('not')"] = bool(re.search(r"\bnot\b|\bnever\b", neg, re.I))
display(pd.DataFrame(checks))
print("Note: DE #4, DE #7 and JA #3 are descriptive statements; a missing modal there is not necessarily an error.")

## 4. Reading the results
*Fill in after running – every statement must match the tables above.*
* Which model is more accurate per language pair (BLEU / chrF) and which keeps the glossary terms.
* Whether numbers, units and the standard name (ISO 26262) are preserved – critical in specifications.
* CPU latency per sentence → feasibility on an ordinary on‑premise server or engineer laptop, with zero data egress.
* Typical errors (terminology, modal verbs / “shall”, word order in long Japanese sentences) → why a glossary and a
  human review step remain necessary.

**Acknowledgements:** Helsinki‑NLP OPUS‑MT (Tiedemann & Thottingal, 2020), Meta NLLB‑200 (NLLB Team, 2022),
HuggingFace `transformers`, `sacrebleu` (Post, 2018). Synthetic sentences and references prepared by Group 81 with AI assistance.